# 🧪 Embeddings_Comparision_BioMarkers

**Objective:** Evaluate and compare the **accuracy** of different pre-trained **embedding models** for the task of **standardizing raw medical biomarker names** to a canonical list.

---

### **Models Tested:**

* [`cambridgeltl/SapBERT-from-PubMedBERT-fulltext`](https://huggingface.co/cambridgeltl/SapBERT-from-PubMedBERT-fulltext) (Specialized Biomedical Entity Linker) **Accuracy: 95.88 %** 🏆
* [`abhinand/MedEmbed-small-v0.1`](https://huggingface.co/abhinand/MedEmbed-small-v0.1) (General Medical Embeddings - Small) **Accuracy: 80.41 %**
* [`google/embeddinggemma-300m`](https://huggingface.co/google/embeddinggemma-300m) (Google's Open Embedding Model) **Accuracy: 79.38 %**
* [`abhinand/MedEmbed-large-v0.1`](https://huggingface.co/abhinand/MedEmbed-large-v0.1) (General Medical Embeddings - Large) **Accuracy: 79.38 %**
* [`dmis-lab/biobert-base-cased-v1.2`](https://huggingface.co/dmis-lab/biobert-base-cased-v1.2) (General Biomedical BERT) **Accuracy: 48.45 %**

---

### **Evaluation:**

Each model is tested against a comprehensive list of raw biomarker names (`V3 Test Suite`) with a known ground truth mapping (`answer_key`). Accuracy is calculated based on the model's ability to identify the correct canonical name or correctly reject irrelevant/ambiguous terms using a **cosine similarity threshold of `0.60`**.

---

#**The test list and answer dictionary**

In [ ]:
# --The Biomarker Lists AND the Answer Key ---
canonical_biomarkers = [
    "High-Density Lipoprotein Cholesterol (HDL-C)", "Low-Density Lipoprotein Cholesterol (LDL-C)",
    "Triglycerides", "Apolipoprotein B (ApoB)", "Lipoprotein(a) (Lp(a))",
    "Homocysteine", "C-Reactive Protein, High-Sensitivity (hs-CRP)",
    "Thyroid-Stimulating Hormone (TSH)", "Thyroxine (T4), Free", "Triiodothyronine (T3), Free",
    "Thyroid Peroxidase Antibodies (TPOAb)", "Dehydroepiandrosterone Sulfate (DHEA-S)",
    "Sex Hormone-Binding Globulin (SHBG)", "Testosterone, Total", "Luteinizing Hormone (LH)",
    "Follicle-Stimulating Hormone (FSH)", "Prostate-Specific Antigen (PSA), Total",
    "Gamma-Glutamyl Transferase (GGT)", "Alanine Aminotransferase (ALT)",
    "Aspartate Aminotransferase (AST)", "Alkaline Phosphatase (ALP)", "Bilirubin, Total",
    "Albumin", "Creatinine", "Blood Urea Nitrogen (BUN)", "Estimated Glomerular Filtration Rate (eGFR)",
    "Creatine Kinase (CK)", "Uric Acid", "Insulin-like Growth Factor 1 (IGF-1)",
    "Insulin", "Hemoglobin A1c (HbA1c)", "25-Hydroxyvitamin D", "Vitamin B12",
    "White Blood Cell Count (WBC)", "Red Blood Cell Count (RBC)", "Hemoglobin (Hgb)",
    "Hematocrit (Hct)", "Platelet Count (PLT)", "Mean Corpuscular Volume (MCV)",
    "Iron, Total", "Total Iron-Binding Capacity (TIBC)", "Transferrin Saturation",
    "Ferritin", "Sodium", "Potassium", "Chloride"
]

# This is your "Ground Truth" Answer Key.
# We map each raw name to its one, correct canonical name.
# For "negative" cases we want to reject, we map them to None.
answer_key = {
    "HDL Cholesterol": "High-Density Lipoprotein Cholesterol (HDL-C)",
    "hdl-c direct": "High-Density Lipoprotein Cholesterol (HDL-C)",
    "good cholesterol": "High-Density Lipoprotein Cholesterol (HDL-C)",
    "LDL, Calculated": "Low-Density Lipoprotein Cholesterol (LDL-C)",
    "ldl direct measurement": "Low-Density Lipoprotein Cholesterol (LDL-C)",
    "bad cholesterol": "Low-Density Lipoprotein Cholesterol (LDL-C)",
    "Trigs": "Triglycerides",
    "triglyceride panel": "Triglycerides",
    "Apo B-100": "Apolipoprotein B (ApoB)",
    "Apolipo B": "Apolipoprotein B (ApoB)",
    "Lp(a) particle number": "Lipoprotein(a) (Lp(a))",
    "Lipoprotein a": "Lipoprotein(a) (Lp(a))",
    "HCY, Plasma": "Homocysteine",
    "Homocystine": "Homocysteine",
    "hsCRP": "C-Reactive Protein, High-Sensitivity (hs-CRP)",
    "C Reactive Protein (Cardiac)": "C-Reactive Protein, High-Sensitivity (hs-CRP)",
    "CRP High Sensitivity": "C-Reactive Protein, High-Sensitivity (hs-CRP)",
    "Free T4": "Thyroxine (T4), Free",
    "FT4 index": "Thyroxine (T4), Free",
    "Thyroxine, Free, Direct": "Thyroxine (T4), Free",
    "Free T3": "Triiodothyronine (T3), Free",
    "FT3": "Triiodothyronine (T3), Free",
    "Total Triiodothyronine, Free": "Triiodothyronine (T3), Free",
    "Anti-TPO": "Thyroid Peroxidase Antibodies (TPOAb)",
    "Thyroid Peroxidase Ab": "Thyroid Peroxidase Antibodies (TPOAb)",
    "TPO Antibodies": "Thyroid Peroxidase Antibodies (TPOAb)",
    "DHEA Sulfate": "Dehydroepiandrosterone Sulfate (DHEA-S)",
    "DHEAS": "Dehydroepiandrosterone Sulfate (DHEA-S)",
    "SHBG, serum": "Sex Hormone-Binding Globulin (SHBG)",
    "Sex Hormone BG": "Sex Hormone-Binding Globulin (SHBG)",
    "Testosterone, T": "Testosterone, Total",
    "Total Testosterone": "Testosterone, Total",
    "LH, Serum": "Luteinizing Hormone (LH)",
    "Luteinising Hormone": "Luteinizing Hormone (LH)",
    "FSH Hormone": "Follicle-Stimulating Hormone (FSH)",
    "Follicular Stimulating Hormone": "Follicle-Stimulating Hormone (FSH)",
    "GGT, serum": "Gamma-Glutamyl Transferase (GGT)",
    "Gamma-Glutamyltranspeptidase": "Gamma-Glutamyl Transferase (GGT)",
    "ALT (SGPT)": "Alanine Aminotransferase (ALT)",
    "Alanine Transaminase": "Alanine Aminotransferase (ALT)",
    "Serum Glutamic-Pyruvic Transaminase": "Alanine Aminotransferase (ALT)",  # Known failure case
    "AST (SGOT)": "Aspartate Aminotransferase (AST)",
    "Aspartate Transaminase": "Aspartate Aminotransferase (AST)",
    "Serum Glutamic-Oxaloacetic Transaminase": "Aspartate Aminotransferase (AST)",
    "CK, Total": "Creatine Kinase (CK)",
    "Creatinine Kinase": "Creatine Kinase (CK)",
    "Urate": "Uric Acid",
    "Serum Uric Acid": "Uric Acid",
    "IGF-1 by LC/MS": "Insulin-like Growth Factor 1 (IGF-1)",
    "Somatomedin C": "Insulin-like Growth Factor 1 (IGF-1)",
    "Vitamin D, 25-Hydroxy": "25-Hydroxyvitamin D",
    "25-OH-D": "25-Hydroxyvitamin D",
    "Creatinine, Serum": "Creatinine",
    "BUN": "Blood Urea Nitrogen (BUN)",
    "Urea Nitrogen, Blood": "Blood Urea Nitrogen (BUN)",
    "GFR, Estimated": "Estimated Glomerular Filtration Rate (eGFR)",
    "Glomerular Filtration Rate, Est.": "Estimated Glomerular Filtration Rate (eGFR)",
    "ALK Phos": "Alkaline Phosphatase (ALP)",
    "Alkaline Phos.": "Alkaline Phosphatase (ALP)",
    "Bilirubin Total": "Bilirubin, Total",
    "Total Bilirubin": "Bilirubin, Total",
    "ALB": "Albumin",
    "WBC Count": "White Blood Cell Count (WBC)",
    "Leukocytes": "White Blood Cell Count (WBC)",
    "RBC": "Red Blood Cell Count (RBC)",
    "Erythrocytes": "Red Blood Cell Count (RBC)",
    "HGB": "Hemoglobin (Hgb)",
    "Haemoglobin": "Hemoglobin (Hgb)",
    "HCT": "Hematocrit (Hct)",
    "Packed Cell Volume (PCV)": "Hematocrit (Hct)",
    "Platelets": "Platelet Count (PLT)",
    "Thrombocytes": "Platelet Count (PLT)",
    "MCV": "Mean Corpuscular Volume (MCV)",
    "Serum Iron": "Iron, Total",  # Known failure case
    "TIBC": "Total Iron-Binding Capacity (TIBC)",
    "Iron Binding Capacity, Total": "Total Iron-Binding Capacity (TIBC)",
    "Transferrin % Saturation": "Transferrin Saturation",
    "TSAT": "Transferrin Saturation",  # Known failure case
    "Ferritin, Serum": "Ferritin",
    "PSA Total": "Prostate-Specific Antigen (PSA), Total",
    "Prostate Specific Ag": "Prostate-Specific Antigen (PSA), Total",
    "Insulin, Fasting": "Insulin",
    "Na+": "Sodium",
    "K+": "Potassium",
    "Cl-": "Chloride",
    "Sodium, Plasma": "Sodium",
    "Potassium, Serum": "Potassium",
    # Panel Names (Should be rejected)
    "Complete Blood Count": None,
    "CBC w/ Differential": None,
    "Comprehensive Metabolic Panel": None,
    "CMP": None,
    "Lipid Panel": None,
    # Deliberate Negative Cases (Should be rejected)
    "Blood Pressure": None,
    "Patient Name": None,
    "Date of Birth": None,
    "Lab Address": None,
    "Fasting Status: Yes": None
}

#**cambridgeltl/SapBERT-from-PubMedBERT-fulltext**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from scipy.spatial.distance import cosine

# --- 1. Load the SPECIALIST Model and Tokenizer ---
print("Loading SapBERT model...")
tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
print("Model loaded successfully.")


# --- 2. Define a function to get embeddings ---
def get_embedding(text, tokenizer, model):
    """Generates an embedding for the given text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :].squeeze()
    return embedding


# --- 3. Pre-calculate embeddings for your canonical list ---
print("\nGenerating embeddings for canonical biomarkers...")
# This part remains the same
canonical_embeddings = {name: get_embedding(name, tokenizer, model) for name in canonical_biomarkers}
print("Embeddings generated.")

# --- 4. Test and Score the model ---
print("\n--- Automated Scoring with SapBERT (V3 Suite) ---")

correct_count = 0
total_count = len(answer_key)
REJECTION_THRESHOLD = 0.60  # Define your rejection threshold

# Loop over the answer key instead of the raw list
for raw_name, correct_answer in answer_key.items():

    raw_embedding = get_embedding(raw_name, tokenizer, model)

    scores = {}
    for canonical_name, canonical_emb in canonical_embeddings.items():
        similarity = 1 - cosine(raw_embedding, canonical_emb)
        scores[canonical_name] = similarity

    best_match_name = max(scores, key=scores.get)
    best_match_score = scores[best_match_name]

    # --- This is the new automated scoring logic ---
    is_correct = False

    if correct_answer is None:
        # Case 1: This item was supposed to be rejected (it's a "None" case)
        if best_match_score < REJECTION_THRESHOLD:
            is_correct = True
    else:
        # Case 2: This item was supposed to match a specific name
        if best_match_name == correct_answer:
            is_correct = True

    # --- Print the result ---
    print(f"Raw Name: '{raw_name}' (Score: {best_match_score:.4f})")

    if is_correct:
        correct_count += 1
        if correct_answer is None:
            print(f"   -> 🟢 CORRECT (Successfully rejected)\n")
        else:
            print(f"   -> 🟢 CORRECT (Matched: {best_match_name})\n")
    else:
        print(f"   -> 🔴 INCORRECT (Expected: {correct_answer}, Got: {best_match_name})\n")

# --- 6. Print Final Score ---
print("\n--- FINAL RESULTS ---")
accuracy = (correct_count / total_count) * 100
print(f"Total Items:     {total_count}")
print(f"Correct Matches: {correct_count}")
print(f"Incorrect:       {total_count - correct_count}")
print(f"Accuracy:        {accuracy:.2f}%")

Loading SapBERT model...


tokenizer_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Model loaded successfully.

Generating embeddings for canonical biomarkers...
Embeddings generated.

--- Automated Scoring with SapBERT (V3 Suite) ---
Raw Name: 'HDL Cholesterol' (Score: 0.9184)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'hdl-c direct' (Score: 0.7147)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'good cholesterol' (Score: 0.6475)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'LDL, Calculated' (Score: 0.7586)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'ldl direct measurement' (Score: 0.6766)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'bad cholesterol' (Score: 0.6271)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'Trigs' (Score: 0.7130)
   -> 🟢 CORRECT (Matched: Triglycerides)

Raw Name: 'triglyceride panel' (Score: 0.7383)
   -> 🟢 CORRECT (Match

#**abhinand/MedEmbed-small-v0.1**

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from scipy.spatial.distance import cosine  # We still use cosine for consistency if util fails, but util is better.

# --- 1. Load the new model ---
print("Loading MedEmbed-small-v0.1 model...")
try:
    model = SentenceTransformer("abhinand/MedEmbed-small-v0.1")
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model: {e}")
    exit()

# --- 2. Pre-calculate embeddings for your canonical list ---
print("\nGenerating embeddings for canonical biomarkers...")
# The model.encode() function is the SentenceTransformer way to get embeddings
canonical_embeddings = model.encode(canonical_biomarkers, convert_to_tensor=True)
print("Embeddings generated.")

# --- 3. Test and Score the model ---
print("\n--- Automated Scoring with MedEmbed-large-v0.1 (V3 Suite) ---")

correct_count = 0
total_count = len(answer_key)
REJECTION_THRESHOLD = 0.60  # We use the same threshold for a fair comparison

for raw_name, correct_answer in answer_key.items():

    # Get the embedding for the raw name
    raw_embedding = model.encode(raw_name, convert_to_tensor=True)

    # Use the SentenceTransformer 'util.cos_sim' for fast similarity calculation
    cosine_scores = util.cos_sim(raw_embedding, canonical_embeddings)

    # Find the best match
    best_match_index = cosine_scores.argmax()
    best_match_name = canonical_biomarkers[best_match_index]
    best_match_score = cosine_scores[0, best_match_index].item()  # .item() gets the Python number

    # --- This is the new automated scoring logic ---
    is_correct = False

    if correct_answer is None:
        # Case 1: This item was supposed to be rejected (it's a "None" case)
        if best_match_score < REJECTION_THRESHOLD:
            is_correct = True
    else:
        # Case 2: This item was supposed to match a specific name
        if best_match_name == correct_answer:
            is_correct = True

    # --- Print the result ---
    print(f"Raw Name: '{raw_name}' (Score: {best_match_score:.4f})")

    if is_correct:
        correct_count += 1
        if correct_answer is None:
            print(f"   -> 🟢 CORRECT (Successfully rejected)\n")
        else:
            print(f"   -> 🟢 CORRECT (Matched: {best_match_name})\n")
    else:
        print(f"   -> 🔴 INCORRECT (Expected: {correct_answer}, Got: {best_match_name})\n")

# --- 5. Print Final Score ---
print("\n--- FINAL RESULTS ---")
accuracy = (correct_count / total_count) * 100
print(f"Total Items:     {total_count}")
print(f"Correct Matches: {correct_count}")
print(f"Incorrect:       {total_count - correct_count}")
print(f"Accuracy:        {accuracy:.2f}%")

Loading MedEmbed-small-v0.1 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded successfully.

Generating embeddings for canonical biomarkers...
Embeddings generated.

--- Automated Scoring with MedEmbed-large-v0.1 (V3 Suite) ---
Raw Name: 'HDL Cholesterol' (Score: 0.9006)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'hdl-c direct' (Score: 0.8542)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'good cholesterol' (Score: 0.7727)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'LDL, Calculated' (Score: 0.8416)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'ldl direct measurement' (Score: 0.8048)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'bad cholesterol' (Score: 0.7871)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'Trigs' (Score: 0.6698)
   -> 🔴 INCORRECT (Expected: Triglycerides, Got: Triiodothyronine (T3), Free)

Raw Name: 'triglycer

#**google/embeddinggemma-300m**

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from scipy.spatial.distance import cosine
import torch

print("Loading EmbeddingGemma-300m model...")
try:
    model = SentenceTransformer("google/embeddinggemma-300m", trust_remote_code=True)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model. Did you accept the terms on Hugging Face? Error: {e}")
    exit()

if torch.cuda.is_available():
    model = model.to('cuda')
    print("Model moved to GPU.")


print("\nGenerating embeddings for canonical biomarkers...")
# The model.encode() function is the SentenceTransformer way to get embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
canonical_embeddings = model.encode(canonical_biomarkers, convert_to_tensor=True, device=device)
print("Embeddings generated.")


# ---  Test and Score the model ---
print("\n--- Automated Scoring with EmbeddingGemma (V3 Suite) ---")

correct_count = 0
total_count = len(answer_key)
REJECTION_THRESHOLD = 0.60

for raw_name, correct_answer in answer_key.items():

    # Get the embedding for the raw name
    raw_embedding = model.encode(raw_name, convert_to_tensor=True, device=device)

    # Use the SentenceTransformer 'util.cos_sim' for fast similarity calculation
    cosine_scores = util.cos_sim(raw_embedding, canonical_embeddings)

    # Find the best match
    best_match_index = cosine_scores.argmax()
    best_match_name = canonical_biomarkers[best_match_index]
    best_match_score = cosine_scores[0, best_match_index].item() # .item() gets the Python number

    is_correct = False

    if correct_answer is None:
        # Case 1: This item was supposed to be rejected (it's a "None" case)
        if best_match_score < REJECTION_THRESHOLD:
            is_correct = True
    else:
        # Case 2: This item was supposed to match a specific name
        if best_match_name == correct_answer:
            is_correct = True

    # --- Print the result ---
    print(f"Raw Name: '{raw_name}' (Score: {best_match_score:.4f})")

    if is_correct:
        correct_count += 1
        if correct_answer is None:
             print(f"   -> 🟢 CORRECT (Successfully rejected)\n")
        else:
             print(f"   -> 🟢 CORRECT (Matched: {best_match_name})\n")
    else:
        print(f"   -> 🔴 INCORRECT (Expected: {correct_answer}, Got: {best_match_name})\n")

# --- 5. Print Final Score ---
print("\n--- FINAL RESULTS ---")
accuracy = (correct_count / total_count) * 100
print(f"Total Items:     {total_count}")
print(f"Correct Matches: {correct_count}")
print(f"Incorrect:       {total_count - correct_count}")
print(f"Accuracy:        {accuracy:.2f}%")

Loading EmbeddingGemma-300m model...
Model loaded successfully.

Generating embeddings for canonical biomarkers...
Embeddings generated.

--- Automated Scoring with EmbeddingGemma (V3 Suite) ---
Raw Name: 'HDL Cholesterol' (Score: 0.7291)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'hdl-c direct' (Score: 0.5395)
   -> 🔴 INCORRECT (Expected: High-Density Lipoprotein Cholesterol (HDL-C), Got: Homocysteine)

Raw Name: 'good cholesterol' (Score: 0.6048)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'LDL, Calculated' (Score: 0.6497)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'ldl direct measurement' (Score: 0.4651)
   -> 🔴 INCORRECT (Expected: Low-Density Lipoprotein Cholesterol (LDL-C), Got: Testosterone, Total)

Raw Name: 'bad cholesterol' (Score: 0.5910)
   -> 🔴 INCORRECT (Expected: Low-Density Lipoprotein Cholesterol (LDL-C), Got: Triglycerides)

Raw Name: 'Trigs' (Score: 0

#**abhinand/MedEmbed-large-v0.1**

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from scipy.spatial.distance import cosine  # We still use cosine for consistency if util fails, but util is better.

# --- 1. Load the new model ---
print("Loading MedEmbed-large-v0.1 model...")
try:
    model = SentenceTransformer("abhinand/MedEmbed-large-v0.1")
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model: {e}")
    exit()

# --- 2. Pre-calculate embeddings for your canonical list ---
print("\nGenerating embeddings for canonical biomarkers...")
# The model.encode() function is the SentenceTransformer way to get embeddings
canonical_embeddings = model.encode(canonical_biomarkers, convert_to_tensor=True)
print("Embeddings generated.")

# --- 3. Test and Score the model ---
print("\n--- Automated Scoring with MedEmbed-large-v0.1 (V3 Suite) ---")

correct_count = 0
total_count = len(answer_key)
REJECTION_THRESHOLD = 0.60  # We use the same threshold for a fair comparison

for raw_name, correct_answer in answer_key.items():

    # Get the embedding for the raw name
    raw_embedding = model.encode(raw_name, convert_to_tensor=True)

    # Use the SentenceTransformer 'util.cos_sim' for fast similarity calculation
    cosine_scores = util.cos_sim(raw_embedding, canonical_embeddings)

    # Find the best match
    best_match_index = cosine_scores.argmax()
    best_match_name = canonical_biomarkers[best_match_index]
    best_match_score = cosine_scores[0, best_match_index].item()  # .item() gets the Python number

    # --- This is the new automated scoring logic ---
    is_correct = False

    if correct_answer is None:
        # Case 1: This item was supposed to be rejected (it's a "None" case)
        if best_match_score < REJECTION_THRESHOLD:
            is_correct = True
    else:
        # Case 2: This item was supposed to match a specific name
        if best_match_name == correct_answer:
            is_correct = True

    # --- Print the result ---
    print(f"Raw Name: '{raw_name}' (Score: {best_match_score:.4f})")

    if is_correct:
        correct_count += 1
        if correct_answer is None:
            print(f"   -> 🟢 CORRECT (Successfully rejected)\n")
        else:
            print(f"   -> 🟢 CORRECT (Matched: {best_match_name})\n")
    else:
        print(f"   -> 🔴 INCORRECT (Expected: {correct_answer}, Got: {best_match_name})\n")

# --- 4. Print Final Score ---
print("\n--- FINAL RESULTS ---")
accuracy = (correct_count / total_count) * 100
print(f"Total Items:     {total_count}")
print(f"Correct Matches: {correct_count}")
print(f"Incorrect:       {total_count - correct_count}")
print(f"Accuracy:        {accuracy:.2f}%")

Loading MedEmbed-large-v0.1 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Model loaded successfully.

Generating embeddings for canonical biomarkers...
Embeddings generated.

--- Automated Scoring with MedEmbed-large-v0.1 (V3 Suite) ---
Raw Name: 'HDL Cholesterol' (Score: 0.8577)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'hdl-c direct' (Score: 0.7963)
   -> 🟢 CORRECT (Matched: High-Density Lipoprotein Cholesterol (HDL-C))

Raw Name: 'good cholesterol' (Score: 0.7432)
   -> 🔴 INCORRECT (Expected: High-Density Lipoprotein Cholesterol (HDL-C), Got: Triglycerides)

Raw Name: 'LDL, Calculated' (Score: 0.7720)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'ldl direct measurement' (Score: 0.7326)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'bad cholesterol' (Score: 0.7367)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'Trigs' (Score: 0.6374)
   -> 🔴 INCORRECT (Expected: Triglycerides, Got: Sodium)

Raw Name: 'triglyc

#**dmis-lab/biobert-base-cased-v1.2**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from scipy.spatial.distance import cosine
import numpy as np

# --- 1. Load the BioBERT Model and Tokenizer ---
print("Loading BioBERT model...")
# Using a popular, cased version of BioBERT
model_name = "dmis-lab/biobert-base-cased-v1.2"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Failed to load model: {e}")
    exit()

# Check if a GPU is available and move the model to it
if torch.cuda.is_available():
    model = model.to('cuda')
    print("Model moved to GPU.")


# --- 2. Define a function to get embeddings ---
def get_embedding(text, tokenizer, model):
    """Generates an embedding for the given text."""
    # Ensure model is on the right device
    device = model.device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # We use mean pooling to get the sentence embedding
    # This averages the vectors of all tokens in the sentence
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze()

    # Detach from GPU and move to CPU for numpy/scipy operations
    return embedding.cpu()


# --- 3. Pre-calculate embeddings for your canonical list ---
print("\nGenerating embeddings for canonical biomarkers...")
canonical_embeddings = {name: get_embedding(name, tokenizer, model) for name in canonical_biomarkers}
print("Embeddings generated.")

# --- 4. Test and Score the model ---
print("\n--- Automated Scoring with BioBERT (V3 Suite) ---")

correct_count = 0
total_count = len(answer_key)
REJECTION_THRESHOLD = 0.60  # We use the same threshold for a fair comparison

for raw_name, correct_answer in answer_key.items():

    raw_embedding = get_embedding(raw_name, tokenizer, model)

    scores = {}
    for canonical_name, canonical_emb in canonical_embeddings.items():
        similarity = 1 - cosine(raw_embedding, canonical_emb)
        scores[canonical_name] = similarity

    best_match_name = max(scores, key=scores.get)
    best_match_score = scores[best_match_name]

    # --- This is the new automated scoring logic ---
    is_correct = False

    if correct_answer is None:
        # Case 1: This item was supposed to be rejected (it's a "None" case)
        if best_match_score < REJECTION_THRESHOLD:
            is_correct = True
    else:
        # Case 2: This item was supposed to match a specific name
        if best_match_name == correct_answer:
            is_correct = True

    # --- Print the result ---
    print(f"Raw Name: '{raw_name}' (Score: {best_match_score:.4f})")

    if is_correct:
        correct_count += 1
        if correct_answer is None:
            print(f"   -> 🟢 CORRECT (Successfully rejected)\n")
        else:
            print(f"   -> 🟢 CORRECT (Matched: {best_match_name})\n")
    else:
        print(f"   -> 🔴 INCORRECT (Expected: {correct_answer}, Got: {best_match_name})\n")

# --- 5. Print Final Score ---
print("\n--- FINAL RESULTS ---")
accuracy = (correct_count / total_count) * 100
print(f"Total Items:     {total_count}")
print(f"Correct Matches: {correct_count}")
print(f"Incorrect:       {total_count - correct_count}")
print(f"Accuracy:        {accuracy:.2f}%")

Loading BioBERT model...


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Model loaded successfully.

Generating embeddings for canonical biomarkers...


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Embeddings generated.

--- Automated Scoring with BioBERT (V3 Suite) ---
Raw Name: 'HDL Cholesterol' (Score: 0.9138)
   -> 🔴 INCORRECT (Expected: High-Density Lipoprotein Cholesterol (HDL-C), Got: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'hdl-c direct' (Score: 0.8574)
   -> 🔴 INCORRECT (Expected: High-Density Lipoprotein Cholesterol (HDL-C), Got: Creatine Kinase (CK))

Raw Name: 'good cholesterol' (Score: 0.9029)
   -> 🔴 INCORRECT (Expected: High-Density Lipoprotein Cholesterol (HDL-C), Got: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name: 'LDL, Calculated' (Score: 0.8460)
   -> 🔴 INCORRECT (Expected: Low-Density Lipoprotein Cholesterol (LDL-C), Got: Total Iron-Binding Capacity (TIBC))

Raw Name: 'ldl direct measurement' (Score: 0.8594)
   -> 🔴 INCORRECT (Expected: Low-Density Lipoprotein Cholesterol (LDL-C), Got: Blood Urea Nitrogen (BUN))

Raw Name: 'bad cholesterol' (Score: 0.9022)
   -> 🟢 CORRECT (Matched: Low-Density Lipoprotein Cholesterol (LDL-C))

Raw Name